# 1. Import and Hardware Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import matplotlib.pyplot as plt
import numpy as np
import copy

!pip install tqdm -q
import tqdm

!pip install wandb -q
import wandb


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [4]:
DATA_PATH = './Data'
SAVE_PATH = '.Model'

In [5]:
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:


KeyboardInterrupt: 

# 2. Hyperparameters

In [ ]:
IMG_SIZE = 224
IN_CHANNELS = 3
BATCH_SIZE = 128

EPOCHS = 300
LR = 0.01
NUM_CLS = 101
DROPOUT_RATE = 0.5

# 3. Training Data Preparation

In [ ]:
# Mean and Std
stats = ((0.545, 0.443, 0.344), (0.269, 0.271, 0.276))

# Transform for train data
train_transform = transforms.Compose([
    transforms.Resize(265),
    transforms.RandomResizedCrop(224),
    transforms.ColorJitter(),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*stats),
    transforms.RandomErasing(p=0.2)
])

# Transform for test data
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [ ]:
# Download the full train data
full_train_data = datasets.Food101(root=DATA_PATH, split='train', 
                        download=True, transform=train_transform)

# Split the full train data into train and validation subset
train_size = int(0.8 * len(full_train_data))
val_size = len(full_train_data) - train_size
train_subset, val_subset = random_split(full_train_data, [train_size, val_size])

# Change the transform of val subset to test_transform
val_subset.dataset = copy.copy(full_train_data)
val_subset.dataset.transform = test_transform

# Download the test data
test_data = datasets.Food101(root=DATA_PATH, split='test', 
                        download=True, transform=test_transform)

In [ ]:
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, 
                        num_workers=4, pin_memory=True, persistent_workers=True)

val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=4, pin_memory=True, persistent_workers=True)

test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=4, pin_memory=True, persistent_workers=True)

# 4. Model Architecture

<table>
  <tr>
    <td><img src="VGG16.png"></td>
    <td><img src="VGG16_2.png"></td>
  </tr>
</table>

In [ ]:
class VGG16(nn.Module):
    def __init__(self, in_channels, num_cls, dropout_rate):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            # ---------- 1. Block ---------- 

            # Conv (224x224x3) -> (224x224x64)
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # Conv (224x224x64) -> (224x224x64)
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # Pool (224x224x64) -> (112x112x64)
            nn.MaxPool2d(kernel_size=2, stride=2),

            # ---------- 2. Block ---------- 

            # Conv (112x112x64) -> (112x112x128)
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # Conv (112x112x128) - (112x112x128)
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # Pool (112x112x128) -> (56x56x128)
            nn.MaxPool2d(kernel_size=2, stride=2),

            # ---------- 3. Block ---------- 
            # (56x56x128) -> (56x56x256)
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # (56x56x256) -> (56x56x256)
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # (56x56x256) -> (56x56x256)
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # (56x56x256) -> (28x28x256)
            nn.MaxPool2d(kernel_size=2, stride=2),

            # ---------- 4. Block ----------
            # (28x28x256) -> (28x28x512) 
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # (28x28x512) -> (28x28x512)
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # (28x28x512) -> (28x28x512)
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # (28x28x512) -> (14x14x512)
            nn.MaxPool2d(kernel_size=2, stride=2),

            # ---------- 5. Block ----------
            # (14x14x512) -> (14x14x512)
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # (14x14x512) -> (14x14x512)
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # (14x14x512) -> (14x14x512)
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # (14x14x512) -> (7x7x512)
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(7*7*512, 4096),
            nn.ReLU(),

            nn.Dropout(dropout_rate),
            nn.Linear(4096, 4096),
            nn.ReLU(),

            nn.Linear(4096, num_cls)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        logits = self.classifier(x)

        return logits

model = VGG16(IN_CHANNELS, NUM_CLS, DROPOUT_RATE).to(device)

# 5. Training Preparation

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, delta = 0, verbose = False, save_path='best_model.pth', trace_func=print):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.save_path = save_path
        self.trace_func = trace_func

        self.best_score = None
        self.earlystop = False
        self.counter = 0

    def __call__(self, val_loss, model):

        # The first epoch
        if self.best_score is None:
            self.best_score = val_loss
            self.save_checkpoint(val_loss, model)

        # The loss didnt reduce much as expect or increased
        elif val_loss >= self.best_score - self.delta:
            self.counter += 1
            self.trace_func(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if (self.counter >= self.patience):
                self.earlystop = True
        
        # The loss reduced properly
        else:
            self.best_score = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        
        if self.verbose:
            self.trace_func("Best Model saving ...")
        torch.save(model.state_dict(), self.save_path)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience= 5, factor=0.1)

# Using GradScaler to prevent Gradient Underflow by using FP16
scaler = torch.apm.GradScaler(device)

In [ ]:
def train(model, loader, criterion, optimizer, scaler):
    # Set the Model in Training Mode
    model.train()

    loop = tqdm(loader, desc="Training", leave=False)
    train_loss, train_acc = 0, 0

    for x, y in loop:
        # Move data to device
        x, y = x.to(device), y.to(device)

        # Clear the Gradient of last batch
        optimizer.zero_grad(set_to_none=True)

        # Get prediction & loss with Automatic Mixed Precision (AMP)
        with torch.amp.autocast(device_type=device.type):
            out = model(x)
            loss = criterion(out, y)

        # Scale up the loss and backpropagate
        scaler.scale(loss).backward()

        # Scale down the Gradients before clipping
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Check if the gradients are valid after Unscaling
        scaler.step(optimizer)

        # Adjust the Scale Factor for the next batch
        scaler.update()

        # Sum up the loss and accuracy
        train_loss += loss.item() * x.size(0)
        train_acc += (out.argmax(1) == y).sum().item()

    return train_loss / len(loader.dataset), train_acc / len(loader.dataset)

In [ ]:
def validate(model, loader, criterion):
    model.eval()
    loop = tqdm(loader, desc="Validation", leave=False)
    val_loss, val_acc = 0, 0

    for x, y in loop:
        x, y = x.to(device), y.to(device)

        with torch.no_grad():
            out = model(x)
            loss = criterion(out, y)

        val_loss += loss.item() * x.size(0)
        val_acc += (out.argmax(1) == y).sum().item()
    
    return val_loss / len(loader.dataset), val_acc / len(loader.dataset)

In [ ]:
def test(model, loader):
    model.eval()
    test_acc = 0
    loop = tqdm(loader, desc="Testing", leave=False)
    for x, y in loop:
        with torch.no_grad():
            out = model(x)
        test_acc += (out.maxarg(1) == y).sum().item()

    return test_acc / len(loader.dataset)

# 6. Training Loop

In [ ]:
wandb.init(
    project="VGG",
    config={
        "Architecture": "VGG16",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMG_SIZE
    }
)

train_accuracies, test_accuracies = [], []
train_losses, val_losses = [], []

early_stopping = EarlyStopping(patience=10)

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, scaler)
    val_loss, val_acc = validate(model,  val_loader, criterion)
    optimizer.step()

    test_acc = test(model, test_loader)

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    wandb.log({
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "lr": optimizer.param_groups[0]['lr']
    })

    print(f"Epoch: {epoch+1}/{EPOCHS} - train_loss: {train_loss}, val_loss: {val_loss}, " + 
           f"train_acc: {train_acc}, val_acc: {val_acc}, test_acc: {test_acc}")

    early_stopping(val_loss, model)

    if early_stopping.earlystop:
        print("Early Stopping")
        break

wandb.finish()

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label = "train_loss")
plt.plot(val_losses, label = "val_loss")
plt.title("Loss History")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label = "train_acc")
plt.plot(test_accuracies, label = "test_acc")
plt.title("Accuracy History")
plt.legend()

# 7. GradCAM

In [ ]:
!pip install grad-cam -q
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import random

# Pick the last Conv Layer to setup the GradCAM
target_layers = [model.feature_extractor[-3]]
cam = GradCAM(model=model, target_layers=target_layers)

# Pick a random image form the test set
imgs, labels = next(iter(test_loader))
index = random.randint(0, len(imgs) - 1)
input_tensor = imgs[index].unsqueeze(0).to(device)
label = labels[index].item()

# Generate the Grad-CAM heatmap
grayscale_cam = cam(input_tensor=input_tensor, targets=None)
grayscale_cam = grayscale_cam[0, :]

# Prepare the image for Visualization
# create a 1D Tensor from the tuple of per-channel means/std
# and reshape it to (3, 1, 1) because imgs[index] has (3, H, W)
mean = torch.tensor(stats[0]).view(3, 1, 1)
std = torch.tensor(stats[1]).view(3, 1, 1)
rgb_img = imgs[index] * std + mean # Denormalize
rgb_img = rgb_img.permute(1, 2, 0).numpy()
rgb_img = np.clip(rgb_img, 0, 1)

# Overlay the heatmap on the image
visual = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

# Get the Prediction
model.eval()
with torch.no_grad():
    out = model(input_tensor)
    pred = out.argmax(1).item()

# 7. Plotting
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title(f"Original (Class: {label})")
plt.imshow(rgb_img)
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title(f"Grad-CAM (Pred: {pred})")
plt.imshow(visualization)
plt.axis('off')
plt.tight_layout()
plt.show()